# 01 Link Elliptic++ to the ledger

1. Matches the labelled Elliptic++ addresses to the Kondor address dictionary.
2. Maps the Elliptic transactions to Kondor transactions, blocks, and timestamps through the deanonymisation map.
3. Derives the block range of each Elliptic time step and the cutoff block, the last block of time step 34.
4. Assigns each labelled address to the training set if it first appears in time steps 1 to 34 and to the test set otherwise.
5. Summarises the training and test periods (Table `tab:elliptic-scope`).

Expected: 265,352 of 265,354 labelled addresses matched, 202,804 of 203,769 Elliptic transactions mapped,
cutoff block 457,734, and 172,903 training and 92,449 test addresses.

In [1]:
from pathlib import Path
import duckdb

ROOT      = Path.cwd().parent
RAW       = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
OUTPUTS   = ROOT / 'outputs'

THREADS      = 192
MEMORY_LIMIT = '320GB'
TEMP_DIR     = PROCESSED / 'duckdb_tmp'

con = duckdb.connect()
con.execute(f'PRAGMA threads={THREADS}')
con.execute(f"PRAGMA memory_limit='{MEMORY_LIMIT}'")
con.execute(f"PRAGMA temp_directory='{TEMP_DIR}'")

import json
import pandas as pd

ELLIPTIC_PP = RAW / 'elliptic_pp'
WALLETS     = ELLIPTIC_PP / 'wallets_features_classes_combined.csv'
TXS         = ELLIPTIC_PP / 'txs_features.csv'
MAP         = RAW / 'kaggle_map' / 'elliptic_deanonymized.csv'

K_ADDRESSES = PROCESSED / 'kondor_addresses.parquet'
K_TXH       = PROCESSED / 'kondor_txh.parquet'
K_TX        = PROCESSED / 'kondor_tx.parquet'
K_BH        = PROCESSED / 'kondor_bh.parquet'

ADDRESS_MAP    = PROCESSED / 'address_map.parquet'
ELLIPTIC_TX    = PROCESSED / 'elliptic_tx.parquet'
TIMESTEP_BLOCK = PROCESSED / 'timestep_blocks.parquet'
CUTOFF         = PROCESSED / 'cutoff.json'
(OUTPUTS / 'tables').mkdir(parents=True, exist_ok=True)

TRAIN_LAST_STEP = 34

## Labelled addresses

The wallet file has one row per address and time step. Classes 1 and 2 are illicit and licit, class 3 is unknown.
Each labelled address carries a single class, and its first time step is the earliest time step in which it appears.

In [2]:
con.execute(f"""
    CREATE OR REPLACE TABLE wallets AS
    SELECT * FROM read_csv('{WALLETS}', header=true)""")

labelled = con.execute("""
    SELECT address, MIN("class") AS label, MAX("class") AS label_check, MIN("Time step") AS first_step
    FROM wallets WHERE "class" IN (1, 2)
    GROUP BY address ORDER BY address""").fetchdf()
assert (labelled.label == labelled.label_check).all()
labelled = labelled.drop(columns='label_check')
print(f'labelled addresses {len(labelled):,}  illicit {(labelled.label == 1).sum():,}  licit {(labelled.label == 2).sum():,}')

labelled addresses 265,354  illicit 14,266  licit 251,088


## Address map

An exact left join on the address string gives each labelled address its Kondor identifier.
Addresses without a match are listed and dropped.

In [3]:
con.register('labelled', labelled)
address_map = con.execute(f"""
    SELECT l.address, l.label, l.first_step, k.addrID
    FROM labelled l LEFT JOIN '{K_ADDRESSES}' k ON l.address = k.address
    ORDER BY l.address""").fetchdf()

unmatched = address_map[address_map.addrID.isna()]
print(f'matched {address_map.addrID.notna().sum():,} of {len(address_map):,}')
print(unmatched[['address', 'label']].to_string(index=False))

address_map = address_map.dropna(subset=['addrID']).astype({'addrID': 'int64'})
address_map['split'] = (address_map.first_step <= TRAIN_LAST_STEP).map({True: 'train', False: 'test'})
address_map.to_parquet(ADDRESS_MAP, index=False)
print(address_map.groupby(['split', 'label']).size().unstack().to_string())

matched 265,352 of 265,354
     address  label
13u4o8S9dnyH      2
  17YCkqFbrp      2
label     1       2
split              
test   4889   87560
train  9377  163526


## Elliptic transactions in the ledger

The map links each Elliptic transaction identifier to its transaction hash. The hash is joined to the Kondor transaction
identifier through the hash dictionary, then to its block through the transaction table, and to the block timestamp through
the block table. Each transaction carries its Elliptic time step from `txs_features.csv`.

In [4]:
tx_map = pd.read_csv(MAP).rename(columns={'txId': 'elliptic_txID', 'transaction': 'tx_hash'})
tx_map['tx_hash'] = tx_map.tx_hash.astype(str).str.strip().str.lower()
steps = pd.read_csv(TXS, usecols=['txId', 'Time step']).rename(
    columns={'txId': 'elliptic_txID', 'Time step': 'time_step'})
con.register('tx_map', tx_map)
con.register('steps', steps)

elliptic_tx = con.execute(f"""
    SELECT s.elliptic_txID, s.time_step, m.tx_hash, h.txID AS kondor_txID, t.blockID, b.block_timestamp
    FROM steps s
    LEFT JOIN tx_map m USING (elliptic_txID)
    LEFT JOIN '{K_TXH}' h ON LOWER(h.tx_hash) = m.tx_hash
    LEFT JOIN '{K_TX}'  t ON t.txID = h.txID
    LEFT JOIN '{K_BH}'  b ON b.blockID = t.blockID
    ORDER BY s.elliptic_txID""").fetchdf()

print(f'Elliptic transactions {len(elliptic_tx):,}')
print(f'in the map            {elliptic_tx.tx_hash.notna().sum():,}')
print(f'found in the ledger   {elliptic_tx.kondor_txID.notna().sum():,}')
elliptic_tx = elliptic_tx.dropna(subset=['kondor_txID']).astype({'kondor_txID': 'int64', 'blockID': 'int64'})
elliptic_tx.to_parquet(ELLIPTIC_TX, index=False)

Elliptic transactions 203,769
in the map            202,804
found in the ledger   202,804


## Time steps on the block axis and the cutoff

In [5]:
blocks = con.execute("""
    SELECT time_step, COUNT(*) AS n_elliptic_txs,
           MIN(blockID) AS first_block, MAX(blockID) AS last_block,
           MIN(block_timestamp) AS first_timestamp, MAX(block_timestamp) AS last_timestamp
    FROM elliptic_tx GROUP BY time_step ORDER BY time_step""").fetchdf()
blocks.to_parquet(TIMESTEP_BLOCK, index=False)

span = blocks.last_block - blocks.first_block + 1
distance = blocks.first_block.iloc[1:].to_numpy() - blocks.last_block.iloc[:-1].to_numpy()
print(f'blocks per time step {span.min()} to {span.max()}')
print(f'first block of a time step minus last block of the previous one {distance.min():,} to {distance.max():,}')

cutoff = int(blocks.loc[blocks.time_step == TRAIN_LAST_STEP, 'last_block'].iloc[0])
next_first = int(blocks.loc[blocks.time_step == TRAIN_LAST_STEP + 1, 'first_block'].iloc[0])
cutoff_ts = int(blocks.loc[blocks.time_step == TRAIN_LAST_STEP, 'last_timestamp'].iloc[0])
CUTOFF.write_text(json.dumps({'cutoff_block': cutoff, 'cutoff_timestamp': cutoff_ts,
                              'train_last_step': TRAIN_LAST_STEP}, indent=2))
print(f'cutoff block {cutoff:,} ({pd.to_datetime(cutoff_ts, unit="s")} UTC), '
      f'{next_first - cutoff - 1:,} blocks before the test period')

blocks per time step 6 to 16
first block of a time step minus last block of the previous one 2,001 to 2,011
cutoff block 457,734 (2017-03-18 01:15:28 UTC), 2,009 blocks before the test period


## Training and test periods

Bitcoin transactions are counted from the block table and include coinbase transactions.
The window also includes the blocks between the two periods, which contain no Elliptic transaction.

In [6]:
periods = {'Training': (1, TRAIN_LAST_STEP), 'Test': (TRAIN_LAST_STEP + 1, 49), 'Window': (1, 49)}
rows = []
for name, (s0, s1) in periods.items():
    b = blocks[(blocks.time_step >= s0) & (blocks.time_step <= s1)]
    first, last = int(b.first_block.min()), int(b.last_block.max())
    n_btc = con.execute(f"SELECT SUM(n_txs) FROM '{K_BH}' WHERE blockID BETWEEN {first} AND {last}").fetchone()[0]
    a = address_map[(address_map.first_step >= s0) & (address_map.first_step <= s1)]
    rows.append({'period': name, 'first_block': first, 'last_block': last,
                 'first_date': pd.to_datetime(b.first_timestamp.min(), unit='s').date(),
                 'last_date': pd.to_datetime(b.last_timestamp.max(), unit='s').date(),
                 'bitcoin_txs': int(n_btc), 'elliptic_txs': int(b.n_elliptic_txs.sum()),
                 'addresses': len(a), 'illicit': int((a.label == 1).sum())})
period_table = pd.DataFrame(rows)
period_table['illicit_share'] = (period_table.illicit / period_table.addresses).round(4)
period_table.to_csv(OUTPUTS / 'tables' / 'periods.csv', index=False)
print(period_table.to_string(index=False))
print(f"Elliptic share of Bitcoin transactions in the window "
      f"{period_table.elliptic_txs.iloc[2] / period_table.bitcoin_txs.iloc[2]:.4%}")

  period  first_block  last_block first_date  last_date  bitcoin_txs  elliptic_txs  addresses  illicit  illicit_share
Training       391200      457734 2016-01-01 2017-03-18    104577292        135864     172903     9377         0.0542
    Test       459744      487975 2017-03-31 2017-10-02     49383660         66940      92449     4889         0.0529
  Window       391200      487975 2016-01-01 2017-10-02    157574585        202804     265352    14266         0.0538
Elliptic share of Bitcoin transactions in the window 0.1287%
